# Avant le TP9 : travailler avec deux clusters

Le TP9 relie deux clusters : **onprem** (`kafka-1` à `kafka-3`) et **cloud** (`kafka-cloud`). Cette démonstration montre, sans MirrorMaker 2 :

1. comment viser l'un ou l'autre cluster depuis Python, et les distinguer ;
2. comment comparer un même compteur des deux côtés : l'offset de fin d'un topic ;
3. comment lire un fichier `.properties` en Python ;
4. comment chercher un motif dans les traces d'un conteneur.

Sorties de référence : exécution du 07/11/2026 22:43 UTC sur le lab.

In [1]:
import os
import time

from confluent_kafka import KafkaException, Producer, TopicPartition
from confluent_kafka.admin import AdminClient, NewTopic, OffsetSpec

BS = os.environ["BS"]
BS_CLOUD = os.environ["BS_CLOUD"]

## 1. Deux clusters, deux clients

Le cluster cloud fait partie du profil Compose `mirror`. On le démarre, puis on attend qu'il réponde.

In [2]:
%%bash
cd $FORMATION_DIR/lab
docker compose --profile mirror up -d kafka-cloud 2>&1

 Volume kafka-lab_kafka-cloud-data Creating 
 Volume kafka-lab_kafka-cloud-data Creating 


 Volume kafka-lab_kafka-cloud-data Created 
 Volume kafka-lab_kafka-cloud-data Created 
 Container k

afka-cloud Creating 


 Container kafka-cloud Created 


 Container kafka-cloud Starting 


 Container kafka-cloud Started 


Un `AdminClient` par cluster : seule l'adresse de départ change. `log_level` 2 coupe le journal de connexion de librdkafka, qui répéterait « Connection refused » tant que le cluster cloud démarre.

In [3]:
onprem = AdminClient({"bootstrap.servers": BS, "log_level": 2})
cloud = AdminClient({"bootstrap.servers": BS_CLOUD, "log_level": 2})

# Attente du cluster cloud : une demande qui echoue est simplement reessayee
for _ in range(45):
    try:
        cloud.describe_cluster(request_timeout=5).result()
        break
    except KafkaException:
        time.sleep(2)

for nom, client in (("onprem", onprem), ("cloud", cloud)):
    description = client.describe_cluster().result()
    print("%-7s cluster_id=%s  noeuds=%s" % (nom, description.cluster_id, sorted(n.id for n in description.nodes)))

onprem  cluster_id=W5FmqobtRp-BMcXlZhVKBw  noeuds=[1, 2, 3]
cloud   cluster_id=tF2HiBQuSD6MqpTGGGBVfA  noeuds=[1]


Les deux clusters ont chacun un nœud d'identifiant 1 : seul l'identifiant de cluster les distingue. `onprem` vise le premier cluster, `cloud` le second.

## 2. Un même compteur des deux côtés

Le nombre de messages écrits dans un topic est la somme des offsets de fin de ses partitions. On crée `demo-tp9` sur onprem seulement, et on le compte des deux côtés.

In [4]:
def fin(client, topic):
    meta = client.list_topics(timeout=10).topics.get(topic)
    if meta is None or meta.error is not None:
        return None
    demandes = {TopicPartition(topic, p): OffsetSpec.latest() for p in meta.partitions}
    return sum(f.result().offset for f in client.list_offsets(demandes).values())


onprem.create_topics([NewTopic("demo-tp9", num_partitions=3, replication_factor=3)])["demo-tp9"].result()
producteur = Producer({"bootstrap.servers": BS, "log_level": 4})
for i in range(500):
    producteur.produce("demo-tp9", key=str(i).encode(), value=b"demo")
producteur.flush(10)
print("onprem :", fin(onprem, "demo-tp9"), "- cloud :", fin(cloud, "demo-tp9"))

onprem : 500 - cloud : None


Côté cloud, le topic n'existe pas : `None`. Au TP, la différence entre les deux côtés sera le **retard de réplication**, en nombre de messages.

## 3. Lire un fichier `.properties`

Les fichiers de configuration du lab sont lisibles depuis le notebook, sous `$FORMATION_DIR/lab` (`FORMATION_DIR` est le dossier de la formation sur votre poste). Exemple avec le nœud cloud :

In [5]:
config = {}
with open(os.environ["FORMATION_DIR"] + "/lab/config/kafka/kafka-cloud/server.properties") as fichier:
    for ligne in fichier:
        ligne = ligne.strip()
        if ligne and not ligne.startswith("#"):
            cle, valeur = ligne.split("=", 1)
            config[cle.strip()] = valeur.strip()
print(len(config), "parametres")
print({cle: config[cle] for cle in ("listeners", "allow.everyone.if.no.acl.found", "default.replication.factor")})

34 parametres
{'listeners': 'CONTROLLER://:9093,INTERNAL://:19092,SECURE://:9096', 'allow.everyone.if.no.acl.found': 'false', 'default.replication.factor': '1'}


## 4. Chercher dans les traces d'un conteneur

In [6]:
traces = !docker compose --project-directory $FORMATION_DIR/lab logs kafka-cloud
print(len(traces), "lignes")
for ligne in traces:
    if "start-cloud" in ligne or "Kafka Server started" in ligne:
        print(ligne[:160])

2219 lignes
kafka-cloud  | [start-cloud] Formatage du stockage (cluster tF2HiBQuSD6MqpTGGGBVfA)
kafka-cloud  | [start-cloud] Lancement de Kafka
kafka-cloud  | [2026-11-07 22:43:16,975] INFO [KafkaRaftServer nodeId=1] Kafka Server started (kafka.server.KafkaRaftServer)


## 5. Ménage

La démonstration supprime son topic et le cluster cloud, avec son volume : au TP, vous le démarrerez vous-même.

In [7]:
onprem.delete_topics(["demo-tp9"])["demo-tp9"].result()

In [8]:
%%bash
cd $FORMATION_DIR/lab
docker compose --profile mirror rm -sfv kafka-cloud >/dev/null 2>&1
docker volume rm kafka-lab_kafka-cloud-data

kafka-lab_kafka-cloud-data


## Méthode pour le TP9

- Deux clients : `onprem` et `cloud`. Toujours se demander **sur quel cluster** on regarde.
- MirrorMaker 2 n'a pas d'API Python : on le démarre par `docker compose`, on lit ses traces, et on juge son travail **à ses effets** sur le cluster cloud.
- Étape 7.3 : un échec est attendu. Lisez les traces avant de corriger.
- Étape 7.5 : écrivez votre prévision avant de lancer la reprise sur le cloud.